# Section 3 — Capacity handling across AUM

**Question:** When a primary futures position exceeds its 2.5% lagged-ADV20 ceiling, which locked residual-allocation rule best tracks the unconstrained 60-session benchmark?

This is Jacopo's final Part 3 analysis. It uses the frozen real Section 0 Phase B handoff, not the earlier synthetic fixture.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / 'config/project.yaml').is_file() and (p / 'data/section0/freeze.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the kernel in the repository or this notebook folder.')
sys.path.insert(0, str(ROOT))

from src.capacity.phase_b import load_capacity_experiment, run_real_capacity_analysis
from src.capacity.analysis import summary_records
from src.validation.schemas import load_project_config

config = load_project_config(ROOT / 'config/project.yaml')
handoff = ROOT / 'data/section0'
experiment = load_capacity_experiment(handoff, config)
print('Loaded frozen Section 0 Phase B handoff.')

## Frozen experiment

The handoff fixes the sample, benchmark, primary contract path, target weights, decision cut-offs, exact-contract valuation, common mask and reference AUM before this analysis runs. Capacity changes residual allocation only. Timing, destination and nearest-half-up rounding remain unchanged.

In [ ]:
import json
freeze = json.loads((handoff / 'freeze.json').read_text())
assert freeze['coverage']['common_valid_dates'] == 148
assert freeze['coverage']['survival'] == '1'
assert freeze['preflight']['references']['capacity'] == 1_000_000_000
assert tuple(freeze['preflight']['core_aum_grid']) == (1_000_000, 10_000_000, 100_000_000, 1_000_000_000)
pprint({'returns': (freeze['frozen_dates'][0], freeze['frozen_dates'][-1]),
        'common_valid_dates': freeze['coverage'],
        'capacity_reference_aum': freeze['preflight']['references']['capacity'],
        'core_aum_grid': freeze['preflight']['core_aum_grid']})

## Methods and fixed choices

The comparison evaluates hard truncate, next-deferred overflow, sequential two-deferred overflow and ADV-weighted two-deferred overflow. Every method uses the same lagged and publication-safe ADV20, 2.5% participation rate, one-day horizon, primary targets, benchmark and post-capacity nearest-half-up rounding. The sole ranking metric is daily-bp Tracking RMSE.

In [ ]:
result = run_real_capacity_analysis(handoff, config)
records = summary_records(result)
reference = result['reference_result']
assert reference['aum'] == Decimal('1000000000')
assert reference['gates']['passed']
assert all(row.common_valid for row in reference['daily'])

for row in records:
    print(row)

## Result and selection

At the frozen $1bn capacity reference, sequential two-deferred overflow has **57.282698 daily bp** Tracking RMSE. ADV-weighted overflow has **57.305883 daily bp**, a difference of 0.023185 bp. That is strictly below the 0.1 bp tie threshold, so the simpler sequential rule is selected.

At $1m, $10m and $100m, capacity does not bind and all four methods produce the same holdings and Tracking RMSE. At $1bn, all 444 primary targets bind, so residual allocation determines the difference.

In [ ]:
selection = reference['selection']
print('Selected rule:', selection['preferred_method'].value)
print('Tied methods:', [method.value for method in selection['tied_methods']])
for row in reference['summary']:
    print(f"{row['method'].value}: {row['tracking_rmse_bps']:.6f} daily bp; "
          f"unimplemented residual ${row['unimplemented_notional']:,.2f}")

## Reproducible outputs

Run `uv run --no-sync python -B scripts/run_person3.py --out outputs/person3` from the repository root to regenerate the CSV outputs, HANDOFF, and charts. `tracking_rmse_vs_aum.png` is the ranking visual. `deferred_exposure_vs_aum.png`, `capacity_allocation_summary.csv` and `capacity_maturity_exposure.csv` show where recovered residual exposure goes; they explain the outcome but do not affect selection.

In [ ]:
from IPython.display import Image, display
figure = ROOT / 'outputs/person3/tracking_rmse_vs_aum.png'
if not figure.exists():
    raise RuntimeError('Run scripts/run_person3.py first to generate the chart.')
display(Image(filename=str(figure)))